In [1]:
from util import create_final_matrix, calculate_global_q, create_master_df
from constants import TEAM_ID_MAP
from helpers import safe_scrape_elo, country_to_country_code, standardise_possessions

ALPHA = 0.1

master_df = create_master_df()
master_df = standardise_possessions(master_df)
global_q, _ = calculate_global_q(master_df)

home = "Brazil"
away = "Norway"

elo_df = safe_scrape_elo()

home_code = country_to_country_code(home)
away_code = country_to_country_code(away)

# print(elo_df.head(20))


# --- SAFE ELO EXTRACTION ---
def get_elo_safely(df, search_code, team_name):
    # Find all rows matching the code
    matches = df.loc[df["country_code"] == search_code, "rating"]

    # Check if the lookup failed
    if len(matches) == 0:
        raise ValueError(
            f"[-] Elo lookup failed: The code '{search_code}' was not found in the database for {team_name}. "
            "Print your elo_df.head(20) to see what string format the TSV is actually using!"
        )

    # If there are duplicates, warn but take the first one
    if len(matches) > 1:
        print(
            f"[!] Warning: Multiple ratings found for {search_code}. Using the first one."
        )

    return float(matches.values[0])


# Run the safe extraction
elo_home = get_elo_safely(elo_df, home_code, home)
elo_away = get_elo_safely(elo_df, away_code, away)


# elo_home = elo_df.loc[elo_df["country_code"] == home_code, "rating"].item()
# elo_away = elo_df.loc[elo_df["country_code"] == away_code, "rating"].item()

print(f"[*] Matchup Set: {home} (ELO {elo_home}) vs {away} (ELO {elo_away})")


final_q_grid = create_final_matrix(
    home,
    TEAM_ID_MAP[home],
    away,
    TEAM_ID_MAP[away],
    elo_home,
    elo_away,
    global_q,
    ALPHA,
)

print(final_q_grid.head())

[*] Scanning './data/world_cup_2026' for match data...
[*] Found 90 match files. Beginning parsing...
  [+] (1/90) Parsed: match_1953853_Mexico_vs_South_Africa.json
  [+] (2/90) Parsed: match_1953854_Mexico_vs_South_Korea.json
  [+] (3/90) Parsed: match_1953855_South_Africa_vs_South_Korea.json
  [+] (4/90) Parsed: match_1953856_Qatar_vs_Switzerland.json
  [+] (5/90) Parsed: match_1953857_Canada_vs_Qatar.json
  [+] (6/90) Parsed: match_1953858_Switzerland_vs_Canada.json
  [+] (7/90) Parsed: match_1953859_Haiti_vs_Scotland.json
  [+] (8/90) Parsed: match_1953860_Brazil_vs_Morocco.json
  [+] (9/90) Parsed: match_1953861_Scotland_vs_Morocco.json
  [+] (10/90) Parsed: match_1953862_Brazil_vs_Haiti.json
  [+] (11/90) Parsed: match_1953863_Scotland_vs_Brazil.json
  [+] (12/90) Parsed: match_1953864_Morocco_vs_Haiti.json
  [+] (13/90) Parsed: match_1953865_USA_vs_Paraguay.json
  [+] (14/90) Parsed: match_1953866_USA_vs_Australia.json
  [+] (15/90) Parsed: match_1953867_Paraguay_vs_Australia.js

In [2]:
from simulator import run_monte_carlo
from helpers import match_outcome_probs_to_odds

prob_home, prob_away, prob_draw = run_monte_carlo(final_q_grid)

home_odds, away_odds, draw_odds = match_outcome_probs_to_odds(
    prob_home, prob_away, prob_draw
)
print(f"{home_odds} | {draw_odds} | {away_odds}")


[*] Prepping matrix for 10000 Monte Carlo simulations...
[*] Running Engine...


Simulating Matches: 100%|██████████| 10000/10000 [05:44<00:00, 29.05match/s]


      FINAL MATCH FORECAST        
Home Win:  0.51%
Draw:      0.20%
Away Win:  0.28%
----------------------------------
Expected Score: Home 2.29 - 1.68 Away

1.945 | 4.936 | 3.531
